# 📘 Fincept Notebook — Financial Statement Ratios

**Finance · Intermediate · ~18 min · pandas + numpy**

Ratios turn raw financial statements into comparable signals about a company's health. We build a compact income statement and balance sheet for a fictional company, *Fincept Demo Corp*, then compute the four classic ratio families — liquidity, profitability, leverage, and efficiency — and interpret each one.

**What you'll learn**
- How to represent an income statement and balance sheet as pandas structures
- Liquidity ratios (current, quick) and what they say about short-term solvency
- Profitability (margins, ROE, ROA) and leverage (debt-to-equity, interest coverage)
- Efficiency via asset turnover, and how to read a tidy ratio table

> **Requires** `pandas` and `numpy` (bundled with Fincept Terminal's Python environment).


## 1. The financial statements

We start with a small, self-contained dictionary of figures (all in $ thousands) for *Fincept Demo Corp*. The income statement flows from revenue down to net income; the balance sheet obeys the identity **Assets = Liabilities + Equity**. We load it into pandas Series so later cells can slice it cleanly. Each code cell re-embeds this dictionary so it can run on its own.


In [ ]:
try:
    import numpy as np
    import pandas as pd
except ImportError:
    raise SystemExit("Fincept Notebook needs pandas + numpy — install with: pip install pandas numpy")

# Fincept Demo Corp — all figures in $ thousands
FIN = {
    # Income statement
    'revenue':            12000.0,
    'cogs':                7200.0,
    'operating_expenses':  2400.0,
    'interest_expense':     300.0,
    'taxes':                540.0,
    # Balance sheet — current assets
    'cash':                1500.0,
    'accounts_receivable': 1800.0,
    'inventory':           2200.0,
    # Balance sheet — non-current assets
    'net_ppe':             6500.0,
    # Balance sheet — liabilities
    'accounts_payable':    1400.0,
    'short_term_debt':      900.0,
    'long_term_debt':      3600.0,
    # Balance sheet — equity
    'common_equity':       6600.0,
}

# Build the income statement as a derived Series
gross_profit = FIN['revenue'] - FIN['cogs']
operating_income = gross_profit - FIN['operating_expenses']
pretax_income = operating_income - FIN['interest_expense']
net_income = pretax_income - FIN['taxes']
income_statement = pd.Series({
    'Revenue': FIN['revenue'],
    'COGS': -FIN['cogs'],
    'Gross profit': gross_profit,
    'Operating expenses': -FIN['operating_expenses'],
    'Operating income (EBIT)': operating_income,
    'Interest expense': -FIN['interest_expense'],
    'Pretax income': pretax_income,
    'Taxes': -FIN['taxes'],
    'Net income': net_income,
}, name='$000s')

current_assets = FIN['cash'] + FIN['accounts_receivable'] + FIN['inventory']
total_assets = current_assets + FIN['net_ppe']
current_liabilities = FIN['accounts_payable'] + FIN['short_term_debt']
total_liabilities = current_liabilities + FIN['long_term_debt']
balance_sheet = pd.Series({
    'Cash': FIN['cash'],
    'Accounts receivable': FIN['accounts_receivable'],
    'Inventory': FIN['inventory'],
    'Total current assets': current_assets,
    'Net PP&E': FIN['net_ppe'],
    'TOTAL ASSETS': total_assets,
    'Current liabilities': current_liabilities,
    'Long-term debt': FIN['long_term_debt'],
    'Total liabilities': total_liabilities,
    'Common equity': FIN['common_equity'],
    'LIAB + EQUITY': total_liabilities + FIN['common_equity'],
}, name='$000s')

print("FINCEPT DEMO CORP — Income Statement ($000s)")
print(income_statement.round(1).to_string())
print()
print("FINCEPT DEMO CORP — Balance Sheet ($000s)")
print(balance_sheet.round(1).to_string())
print()
print(f"Balance check: Assets {total_assets:,.0f} == Liab+Equity {total_liabilities + FIN['common_equity']:,.0f}")


## 2. Liquidity ratios

Liquidity ratios ask: *can the company cover its short-term bills?*

- **Current ratio** = current assets / current liabilities. Above 1.0 means current assets exceed current obligations.
- **Quick (acid-test) ratio** = (current assets − inventory) / current liabilities. It strips out inventory, the least-liquid current asset, for a stricter test.


In [ ]:
try:
    import numpy as np
    import pandas as pd
except ImportError:
    raise SystemExit("Fincept Notebook needs pandas + numpy — install with: pip install pandas numpy")

FIN = {
    'revenue': 12000.0, 'cogs': 7200.0, 'operating_expenses': 2400.0,
    'interest_expense': 300.0, 'taxes': 540.0,
    'cash': 1500.0, 'accounts_receivable': 1800.0, 'inventory': 2200.0,
    'net_ppe': 6500.0, 'accounts_payable': 1400.0, 'short_term_debt': 900.0,
    'long_term_debt': 3600.0, 'common_equity': 6600.0,
}

current_assets = FIN['cash'] + FIN['accounts_receivable'] + FIN['inventory']
current_liabilities = FIN['accounts_payable'] + FIN['short_term_debt']

current_ratio = current_assets / current_liabilities
quick_ratio = (current_assets - FIN['inventory']) / current_liabilities

liquidity = pd.DataFrame({
    'Ratio': ['Current ratio', 'Quick ratio'],
    'Value': [current_ratio, quick_ratio],
    'Formula': ['CA / CL', '(CA - Inventory) / CL'],
    'Interpretation': [
        'Comfortable short-term cushion (>1.5 is healthy)',
        'Can meet current bills without selling inventory',
    ],
})
liquidity['Value'] = liquidity['Value'].round(2)
print("LIQUIDITY RATIOS — Fincept Demo Corp")
print(liquidity.to_string(index=False))


## 3. Profitability ratios

Profitability ratios measure how much of each revenue dollar (or each dollar of capital) becomes profit.

- **Gross / operating / net margin** = gross profit, EBIT, or net income divided by revenue.
- **ROA** = net income / total assets — return generated per dollar of assets.
- **ROE** = net income / common equity — return to shareholders.


In [ ]:
try:
    import numpy as np
    import pandas as pd
except ImportError:
    raise SystemExit("Fincept Notebook needs pandas + numpy — install with: pip install pandas numpy")

FIN = {
    'revenue': 12000.0, 'cogs': 7200.0, 'operating_expenses': 2400.0,
    'interest_expense': 300.0, 'taxes': 540.0,
    'cash': 1500.0, 'accounts_receivable': 1800.0, 'inventory': 2200.0,
    'net_ppe': 6500.0, 'accounts_payable': 1400.0, 'short_term_debt': 900.0,
    'long_term_debt': 3600.0, 'common_equity': 6600.0,
}

gross_profit = FIN['revenue'] - FIN['cogs']
operating_income = gross_profit - FIN['operating_expenses']
pretax_income = operating_income - FIN['interest_expense']
net_income = pretax_income - FIN['taxes']
total_assets = FIN['cash'] + FIN['accounts_receivable'] + FIN['inventory'] + FIN['net_ppe']

profitability = pd.DataFrame({
    'Ratio': ['Gross margin', 'Operating margin', 'Net margin', 'Return on assets (ROA)', 'Return on equity (ROE)'],
    'Value': [
        gross_profit / FIN['revenue'],
        operating_income / FIN['revenue'],
        net_income / FIN['revenue'],
        net_income / total_assets,
        net_income / FIN['common_equity'],
    ],
    'Interpretation': [
        'Share of revenue left after direct costs',
        'Profitability of core operations (pre-interest/tax)',
        'Bottom-line profit per revenue dollar',
        'Profit generated per dollar of assets',
        'Profit generated per dollar of shareholder equity',
    ],
})
profitability['Value %'] = (profitability['Value'] * 100).round(1)
profitability = profitability[['Ratio', 'Value %', 'Interpretation']]
print("PROFITABILITY RATIOS — Fincept Demo Corp (all shown as %)")
print(profitability.to_string(index=False))


## 4. Leverage & efficiency ratios

**Leverage** ratios gauge how much the company relies on debt, and how easily it services it:

- **Debt-to-equity** = total debt / common equity.
- **Interest coverage** = EBIT / interest expense — how many times operating profit covers interest.

**Efficiency** ratios show how hard the asset base works:

- **Asset turnover** = revenue / total assets — revenue produced per dollar of assets.


In [ ]:
try:
    import numpy as np
    import pandas as pd
except ImportError:
    raise SystemExit("Fincept Notebook needs pandas + numpy — install with: pip install pandas numpy")

FIN = {
    'revenue': 12000.0, 'cogs': 7200.0, 'operating_expenses': 2400.0,
    'interest_expense': 300.0, 'taxes': 540.0,
    'cash': 1500.0, 'accounts_receivable': 1800.0, 'inventory': 2200.0,
    'net_ppe': 6500.0, 'accounts_payable': 1400.0, 'short_term_debt': 900.0,
    'long_term_debt': 3600.0, 'common_equity': 6600.0,
}

gross_profit = FIN['revenue'] - FIN['cogs']
operating_income = gross_profit - FIN['operating_expenses']   # EBIT
total_assets = FIN['cash'] + FIN['accounts_receivable'] + FIN['inventory'] + FIN['net_ppe']
total_debt = FIN['short_term_debt'] + FIN['long_term_debt']

debt_to_equity = total_debt / FIN['common_equity']
interest_coverage = operating_income / FIN['interest_expense']
asset_turnover = FIN['revenue'] / total_assets

lev_eff = pd.DataFrame({
    'Family': ['Leverage', 'Leverage', 'Efficiency'],
    'Ratio': ['Debt-to-equity', 'Interest coverage', 'Asset turnover'],
    'Value': [debt_to_equity, interest_coverage, asset_turnover],
    'Unit': ['x', 'x', 'x'],
    'Interpretation': [
        'Debt is below equity — moderate leverage',
        'EBIT covers interest many times over — low default risk',
        'Each $1 of assets generates this much revenue',
    ],
})
lev_eff['Value'] = lev_eff['Value'].round(2)
print("LEVERAGE & EFFICIENCY RATIOS — Fincept Demo Corp")
print(lev_eff.to_string(index=False))


## 5. Full ratio summary

Finally, we assemble every ratio into one tidy DataFrame grouped by family — the kind of one-glance scorecard an analyst keeps next to the statements.


In [ ]:
try:
    import numpy as np
    import pandas as pd
except ImportError:
    raise SystemExit("Fincept Notebook needs pandas + numpy — install with: pip install pandas numpy")

FIN = {
    'revenue': 12000.0, 'cogs': 7200.0, 'operating_expenses': 2400.0,
    'interest_expense': 300.0, 'taxes': 540.0,
    'cash': 1500.0, 'accounts_receivable': 1800.0, 'inventory': 2200.0,
    'net_ppe': 6500.0, 'accounts_payable': 1400.0, 'short_term_debt': 900.0,
    'long_term_debt': 3600.0, 'common_equity': 6600.0,
}

gross_profit = FIN['revenue'] - FIN['cogs']
operating_income = gross_profit - FIN['operating_expenses']
pretax_income = operating_income - FIN['interest_expense']
net_income = pretax_income - FIN['taxes']
current_assets = FIN['cash'] + FIN['accounts_receivable'] + FIN['inventory']
total_assets = current_assets + FIN['net_ppe']
current_liabilities = FIN['accounts_payable'] + FIN['short_term_debt']
total_debt = FIN['short_term_debt'] + FIN['long_term_debt']

rows = [
    ('Liquidity',     'Current ratio',     current_assets / current_liabilities,            'x'),
    ('Liquidity',     'Quick ratio',       (current_assets - FIN['inventory']) / current_liabilities, 'x'),
    ('Profitability', 'Gross margin',      gross_profit / FIN['revenue'] * 100,             '%'),
    ('Profitability', 'Operating margin',  operating_income / FIN['revenue'] * 100,         '%'),
    ('Profitability', 'Net margin',        net_income / FIN['revenue'] * 100,               '%'),
    ('Profitability', 'ROA',               net_income / total_assets * 100,                 '%'),
    ('Profitability', 'ROE',               net_income / FIN['common_equity'] * 100,         '%'),
    ('Leverage',      'Debt-to-equity',    total_debt / FIN['common_equity'],               'x'),
    ('Leverage',      'Interest coverage', operating_income / FIN['interest_expense'],      'x'),
    ('Efficiency',    'Asset turnover',    FIN['revenue'] / total_assets,                   'x'),
]
summary = pd.DataFrame(rows, columns=['Family', 'Ratio', 'Value', 'Unit'])
summary['Value'] = summary['Value'].round(2)
print("FINCEPT DEMO CORP — RATIO SCORECARD")
print(summary.to_string(index=False))


---
*— Fincept Notebook · part of Fincept Terminal. Edit any cell and press Ctrl+Enter to run.*
